# Distributed Training

### Load Data

In [4]:
import pandas as pd

In [5]:
DATASET_LOC = "https://raw.githubusercontent.com/GokuMohandas/Made-With-ML/main/datasets/dataset.csv"
train_df = pd.read_csv(DATASET_LOC)
train_df.head()

,id,created_on,title,description,tag
0,6,2020-02-20 06:43:18,Comparison between YOLO and RCNN on real world...,Bringing theory to experiment is cool. We can ...,computer-vision
1,7,2020-02-20 06:47:21,"Show, Infer & Tell: Contextual Inference for C...",The beauty of the work lies in the way it arch...,computer-vision
2,9,2020-02-24 16:24:45,Awesome Graph Classification,"A collection of important graph embedding, cla...",other
3,15,2020-02-28 23:55:26,Awesome Monte Carlo Tree Search,A curated list of Monte Carlo tree search pape...,other
4,25,2020-03-07 23:04:31,AttentionWalk,"A PyTorch Implementation of ""Watch Your Step: ...",other


In [6]:
train_df.shape

(764, 5)

In [7]:
# Unique Labels
tags = train_df["tag"].unique().tolist()
tags

['computer-vision', 'other', 'natural-language-processing', 'mlops']

In [8]:
HOLDOUT_LOC = "https://raw.githubusercontent.com/GokuMohandas/Made-With-ML/main/datasets/holdout.csv"
test_df = pd.read_csv(HOLDOUT_LOC)

### Utilities


In [9]:
import matplotlib.pyplot as plt
import json
from collections import Counter
import seaborn as sns; sns.set_theme()
from sklearn.metrics import precision_recall_fscore_support
import time
from tqdm import tqdm
import torch
from transformers import pipeline, Trainer, TrainingArguments, DistilBertTokenizer, DistilBertForSequenceClassification

In [29]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [10]:
class_to_index = {label: i for i, label in enumerate(tags)}
index_to_class = {i : label for i, label in enumerate(tags)}

In [18]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english",
                                                            id2label = index_to_class,
                                                            label2id = class_to_index,
                                                            ignore_mismatched_sizes=True)

inputs = tokenizer(train_df["title"][1] + " " + train_df["description"][1], return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased-finetuned-sst-2-english and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([4]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


'mlops'

In [20]:
def get_tag(model, text, tokenizer):
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_class_id = logits.argmax().item()
    return model.config.id2label[predicted_class_id]    

In [23]:
text = train_df["title"][0] + " " + train_df["description"][0]
get_tag(model, text, tokenizer)


'mlops'

In [24]:
# list of dicts with (title, description)
samples = test_df[["title", "description"]].to_dict(orient="records")[:3]
samples

[{'title': 'Diffusion to Vector',
  'description': 'Reference implementation of Diffusion2Vec (Complenet 2018) built on Gensim and NetworkX. '},
 {'title': 'Graph Wavelet Neural Network',
  'description': 'A PyTorch implementation of "Graph Wavelet Neural Network" (ICLR 2019) '},
 {'title': 'Capsule Graph Neural Network',
  'description': 'A PyTorch implementation of "Capsule Graph Neural Network" (ICLR 2019).'}]

In [25]:
# get a list of predictions

def get_predictions(model, text, tokenizer):
    y_pred = []
    for item in tqdm(text):
        input = str(item)
        predicted_tag = get_tag(model, input, tokenizer)

        while predicted_tag is None:
            time.sleep(30)
            predicted_tag = get_tag(model, input, tokenizer)

        y_pred.append(predicted_tag)

    return y_pred

In [26]:
get_predictions(model, samples, tokenizer)

100%|██████████| 3/3 [00:00<00:00, 32.43it/s]


['natural-language-processing',
 'natural-language-processing',
 'natural-language-processing']

In [27]:
test_df.head(3)

,id,created_on,title,description,tag
0,19,2020-03-03 13:54:31,Diffusion to Vector,Reference implementation of Diffusion2Vec (Com...,other
1,26,2020-03-07 23:11:58,Graph Wavelet Neural Network,"A PyTorch implementation of ""Graph Wavelet Neu...",other
2,44,2020-03-08 00:32:58,Capsule Graph Neural Network,"A PyTorch implementation of ""Capsule Graph Neu...",other


### Setup

In [30]:
import os
import random
from ray.data.preprocessor import Preprocessor
import numpy as np

In [31]:
def set_seeds(seed = 42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    eval("setattr(torch.backends.cudnn, 'deterministic', True)")
    eval("setattr(torch.backends.cudnn, 'benchmark, False')")
    os.environ["PYTHONHASHSEED"] = str(seed)


In [32]:
def load_data(num_samples = None):
    ds = ray.data.read_csv(DATASET_LOC)
    ds = ds.random_shuffle(seed=1234)
    ds = ray.data.from_item(ds.take(num_samples)) if num_samples else ds
    return ds

In [33]:
class CustomPreprocessor(Preprocessor):
    """Custom Preprocessor class."""
    def _fit(self, ds):
        tags = ds.unique(column="tag")
        self.class_to_index = {tag: i for i, tag in enumerate(tags)}
        self.index_to_class = {i: tag for i, tag in enumerate(tags)}

    def _trasform_pandas(self, batch):
        return preprocess(batch, class_to_index = self.class_to_index)

### Model

In [34]:
import torch.nn as nn
from transformers import BertModel

In [35]:
llm = BertModel.from_pretrained("allenai/scibert_scivocab_uncased", return_dict = False)
embedding_dim = llm.config.hidden_size
embedding_dim

/home/krekken/madewithml/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Some weights of the model checkpoint at allenai/scibert_scivocab_uncased were not used when initializing BertModel: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


768

In [37]:
# Tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
text = "Single cell RNA-seq reveals distinct cell types."
tokens = tokenizer.tokenize(text)
input_ids = tokenizer.encode(text, return_tensors="pt")
input_ids

tensor([[  102,  1232,   377,  2980,   579, 26317,  8234,  3646,   377,  1910,
           205,   103]])

In [43]:
text = "Transfer learning with transformers for text classification."
batch = tokenizer([text], return_tensors = "pt", padding="longest")
batch

{'input_ids': tensor([[  102,  2268,  1904,   190, 29155,   168,  3267,  2998,   205,   103]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [44]:
batch["input_ids"]

tensor([[  102,  2268,  1904,   190, 29155,   168,  3267,  2998,   205,   103]])

In [45]:
seq,pool = llm(input_ids=batch["input_ids"], attention_mask = batch["attention_mask"])
seq.size(), pool.size()

(torch.Size([1, 10, 768]), torch.Size([1, 768]))

In [49]:
# Finetuning
class FinetunedLLM(nn.Module):
    def __init__(self, llm, dropout_p, embedding_dim, num_classes):
        super(FinetunedLLM, self).__init__()
        self.llm = llm
        self.dropout = nn.Dropout(dropout_p)
        self.fc1 = nn.Linear(embedding_dim, num_classes)

    def forward(self, batch):
        ids, masks = batch["ids"], batch["masks"]
        seq, pool = self.llm(input_ids = ids, attention_mask=masks)
        z = self.dropout(pool)
        z = self.fc1(z)
        return z
    @torch.inference_mode()
    def predict(self, batch):
        self.eval()
        z = self(batch)
        y_pred = torch.argmax(z, dim=1).cpu().numpy()
        return y_pred
    
    @torch.inference_mode()
    def predict_proba(self, batch):
        self.eval()
        z = self(batch)
        y_probs = F.softmax(z).cpu().numpy()
        return y_probs


In [50]:
# Initialize model
model = FinetunedLLM(llm = llm, dropout_p=0.5, embedding_dim=embedding_dim, num_classes=4)
model.named_parameters

<bound method Module.named_parameters of FinetunedLLM(
  (llm): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31090, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

### Batching

In [51]:
from ray.train.torch import get_device